# SAND (Spatial Adjacency Network Detection)
SAND identifies local structures ('clusters') in spatial data based on user defined values


In [1]:
import pandas as pd
import pickle
import anndata as ad
from scipy.cluster.hierarchy import DisjointSet
from sklearn.neighbors import NearestNeighbors
import scipy

In [2]:
"""
SAND: identification of local structures in spatial data
Inputs  1) adata: anndata object with cell type annotations under .obs['ct'], and x,y spatial coordinates under .obsm['spatial']
        2) k: number of nearest neighbors for KNN graph
        3) r: distance cutoff between two cells to filter KNN graph when k is defined or for the construction of a radius-based graph
        4) valuelist: list of cell types of interest (optional)
        
Returns 1) Annotates adata object input with cluster assignments in .obs['cluster']
"""
def runSAND(adata, k = None, r = None, valuelist = []):
    if k is None and r is None:
        raise ValueError("Define at least one: k, r")
 
    filtered_neighs = makeneighborgraph(adata, k, r, valuelist)
    joinsets(adata, filtered_neighs)
    if len(valuelist):
        adata.obs.loc[~adata.obs['ct'].isin(valuelist), 'cluster'] = 'NA'
    
def makeneighborgraph(adata, k, r, valuelist):
    keepidx = [i for i in range(adata.shape[0])]
    if len(valuelist):
        keepidx = [i for i,c in enumerate(adata.obs['ct']) if c in valuelist]              
    map_idxtoct = dict(map(lambda i,j : (i,j) , keepidx,[adata.obs['ct'][x] for x in keepidx])) 
    
    if k != None: # KNN graph
        knn = NearestNeighbors(n_neighbors=k)
        knn.fit(adata.obsm['spatial'])
        neighbors = knn.kneighbors()

        r_filter = r
        if r is None:
            r_filter = float('inf')
        
        filtered_neighs = dict() 
        for idx in keepidx:
            filtered_neighs[idx] = set([
                j for i,j in enumerate(neighbors[1][idx])
                if (j in map_idxtoct) and (neighbors[0][idx][i] < r_filter) # filter neighbors by cell type and radius
            ]) 
    
        return filtered_neighs
    
    else: # radius graph
        tree = scipy.spatial.KDTree(adata.obsm['spatial']) 
        neighbors = tree.query_ball_tree(tree, r) 
    
        filtered_neighs = dict()
        for idx in keepidx:
            filtered_neighs[idx] = set([
                j for j in neighbors[idx]
                if j in map_idxtoct and j != idx
            ]) 
            
        return filtered_neighs

def joinsets(adata, filtered_neighs): # call union find
    edges = set()
    nodes = set()
    for k in filtered_neighs:
        nodes.add(k)
        for v in filtered_neighs[k]:
            edges.add((k,v))
            
    disjoint_set = DisjointSet(nodes)
    for a,b in edges:
        disjoint_set.merge(a, b)

    clusters = dict() # key is cell; value is cluster assignment (root cell)
    for s in disjoint_set.subsets():
        for c in s:
            clusters[c] = disjoint_set.__getitem__(c)

    adata.obs['cluster'] = range(adata.shape[0])
    adata.obs = adata.obs.replace({"cluster": clusters})
    

In [3]:
adata = ad.read_h5ad('/data1/greenbab/users/zhangb2/xenium_files/review_samples/toy_example/xenium_test_data.h5ad')

In [4]:
adata.obs

,sample_name,ct,study_id,diagnosis
aaaaojnj-1-0,region1_0011549,Erythroid,MDS-8,MDS-LB
aaaapmda-1-0,region1_0011549,Late/Mature Myeloid,MDS-8,MDS-LB
aaabbcdp-1-0,region1_0011549,T Cell,MDS-8,MDS-LB
aaabkmdn-1-0,region1_0011549,Erythroid,MDS-8,MDS-LB
aaabmffo-1-0,region1_0011549,Erythroid,MDS-8,MDS-LB
...,...,...,...,...
mega-1555-54,region1_0069665,Megakaryocyte,MDS-31,MDS-IB1
mega-1556-54,region1_0069665,Megakaryocyte,MDS-31,MDS-IB1
mega-1557-54,region1_0069665,Megakaryocyte,MDS-31,MDS-IB1
mega-1558-54,region1_0069665,Megakaryocyte,MDS-31,MDS-IB1


In [5]:
adata.obsm['spatial']

array([[ 8052.32246051,  5706.30418283],
       [ 8289.39414416, 11536.8386676 ],
       [ 8572.75282306,  4194.90862459],
       ...,
       [ 6804.583042  ,  4607.2823741 ],
       [ 6750.38585706,  3858.30529256],
       [ 6213.09280521,  4594.49415232]])

In [6]:
# Run SAND for each sample to find immune aggregates
sand_di = dict()
for samp in adata.obs['sample_name'].unique():
    print(samp)
    ad = adata[adata.obs['sample_name'] == samp].copy()
    samp = str(samp)
    runSAND(ad, 10, 15, ['T Cell', 'B Cell', 'pDC', 'NK'])
    for i in range(ad.shape[0]):
        clustid = str(ad.obs['cluster'][i])
        if clustid != 'NA':
            clustid = clustid + '_' + samp
        sand_di[ad.obs.index[i]] = clustid

adata.obs['cluster'] = [sand_di[x] for x in adata.obs.index]

region1_0011549
region1_0027860
region2_0027861
region1_0069665


In [7]:
# Size of immune aggregates
adata.obs['cluster'].value_counts()

NA                        443510
138685_region1_0069665       243
38475_region1_0069665        238
3039_region1_0011549         204
166795_region1_0011549       177
                           ...  
37218_region1_0027860          1
37204_region1_0027860          1
37197_region1_0027860          1
37174_region1_0027860          1
177802_region1_0069665         1
Name: cluster, Length: 28926, dtype: int64

In [8]:
# Cell types within largest immune aggregate
adata.obs['ct'][adata.obs['cluster'] == '138685_region1_0069665'].value_counts()

B Cell                 136
T Cell                  97
pDC                      9
NK                       1
Megakaryocyte            0
VSMC                     0
Plasma Cell              0
Osteoblast               0
Mesenchymal/Stromal      0
Adipocyte                0
Macrophage               0
Late/Mature Myeloid      0
HSPC                     0
Erythroid                0
Endothelial              0
Early Myeloid            0
Mast                     0
Name: ct, dtype: int64